# Práctica 4. Procesamiento audio - Aplicación de filtros

## Importación de librerías

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import (
    butter, cheby1, cheby2, filtfilt, lfilter, firwin
)
from scipy.fft import rfft, rfftfreq
import streamlit as st
from streamlit_webrtc import webrtc_streamer, WebRtcMode, AudioProcessorBase
import av
import queue
import threading
from st_audiorec import st_audiorec
import soundfile as sf

## Explicación del código añadido
En la aplicación se divide en 2 secciones principales, una donde se declaran todas las funciones necesarias para el procesamiento de audio y otra donde se crea la interfaz de usuario con Streamlit.

### Funciones de procesamiento de audio
Las funciones utilizadas son:
- `to_mono`: Convierte una señal estéreo(dos canales) a mono (un canal) promediando ambos canales.
- `normalize`: Normaliza la amplitud de las eñal para que su valor abosluto sea 1, evitando saturaciones.
- `add_white_noise`: Añade ruido blanco gaussiano a una señal según un SNR (Signal to Noise Ratio) especificado. Calcula la potencia de la señal, luego calcula la potencia del ruido necesaria para alcanzar el SNR deseado y genera el ruido blanco, aplicando luego normalización.
- `make_note` : Genera una señal sintética armónica compuesta de una suma de senos que representan una nota musical, define un eje temporal y genera la onda fundamental y sus armónicos, por último normaliza la señal.
- `safe_cutoffs`: Asegura que las frecuencias de corte para los filtros estén dentro del rango válido (0, fs/2). Esto evita errores númericos y aliasing (efecto de plegado espectral) al diseñar los filtros.
- `design_filter`: Diseña el filtro según el tipo e implementación seleccionados, siendo el tipo de respuesta (pasa-bajo, pasa-alto, pasa-banda, rechaza-banda) y la implementación (FIR o IIR, incluyendo Butterworth, Chebyshev I, Chebyshev II).
    1. Se normaliza la frecuencia de corte en el rango [0, 1] dividiéndola por fs/2, esto se hace porque las funciones de diseño de filtros en scipy esperan frecuencias normalizadas.
    2. Según el tipo de filtro seleccionado, se define la respuesta en frecuencia (lowpass, highpass, bandpass, bandstop).
        - `Pasa-Bajo` : Deja pasar frecuencias por debajo de la frecuencia de corte.
        - `Pasa-Alto` : Deja pasar frecuencias por encima de la frecuencia de corte.
        - `Pasa-Banda` : Deja pasar frecuencias dentro de un rango definido por dos frecuencias de corte.
        - `Rechaza-Banda` : Atenúa frecuencias dentro de un rango definido por dos frecuencias de corte.
    3. Se tiene que seleccionar dos de los siguientes diseños:

        - FIR se genera un filtro de respuesta finita al impulso utilizando las diferentes ventanas (hamming, hann, blackman, bartlett), este diseño funciona aplicando la función y esto es finito en el tiempo, debido a que el filtro promedia las muestras con distintos pesos. Se utiliza la función `firwin` de scipy para diseñar el filtro FIR con los parámetros especificados, que son el número de taps, las frecuencias de corte normalizadas y la ventana seleccionada.

        - IIR se generan filtros de respuesta infinita al impulso utilizando los métodos Butterworth, Chebyshev I y Chebyshev II. Estos filtros tienen una respuesta al impulso que dura indefinidamente, ya que dependen de las muestras pasadas y porque se retroalimentan de las propias salidas.

            - Butterworth: Diseña un filtro Butterworth utilizando la función `butter` de scipy, que proporciona una respuesta en frecuencia suave y sin ondulaciones en la banda pasante. S
            - Chebyshev I: Diseña un filtro Chebyshev tipo I utilizando la función `cheby1` de scipy, que permite un ripple (rizado) controlado en la banda pasante. El rizado se controla con el parametro `rp`, lo que hace es que la respuesta en frecuencia tenga pequeñas ondulaciones dentro de la banda pasante.
            - Chebyshev II: Diseña un filtro Chebyshev tipo II utilizando la función `cheby2` de scipy, que permite un ripple (rizado) controlado en la banda de rechazo. El rizado se controla con el parametro `rs`, lo que hace es que la respuesta en frecuencia tenga pequeñas ondulaciones dentro de la banda de rechazo.
        
        Es importante tener un `order` adecuado para el filtro, ya que el orden lo que dice es la pendiente del corte que indica que tan rápido baja la ganancia desde el punto de corte.
- `apply_filter`: Aplica el filtro que se haya seleccionado utilizando las funciones `safe_cutoffs` que se encarga dde comprobar que frecuencias están dentro de los límites y `design_filter` que diseña el filtro según los parámetros seleccionados, esta función devuelve `a` que controla la parte de la salida pasada (solo en IIR), se basa en que mezcla la señal que ya salío de nuevo y `b` que controla la parte de la entrada, que se encarga de ver las muestras originales. Luego se elige como aplicar el filtro, que se puede hacer con diferentes métodos:

    - `filtfilt`: Aplica el filtro hacia adelante y luego hacia atrás para evitar desfases de fase. Esto es útil cuando se desea preservar la forma de la señal original.

    - `lfilter`: Aplica el filtro de manera causal, lo que puede introducir un desfase en la señal filtrada, esto es meterle un retardo a la señal y se usa para funcionamiento en tiempo real.

    Por último se normaliza la señal filtrada para evitar saturaciones.

- `compute_fft_pos` : Esta función actua para saber que notas hay dentro de la señal y cuán fuerte son. Primero calcula la FTT completa (frecuecnias positivas y negativas), lugeo genera el eje de frecuencias asociado, desde -fs/2 hasta +fs/2, como la señal es real solo se necesita el especto positivo y por último se toma la magnitud sindesface para graficarla mejor.

- `plot_signals_like_example` : Esta función se encarga de graficar la señal antes y después del filtrado.  
    - Las primeras dos gráficas muestran como se ve la señal antes (ruidosa) y después (filtrada) a lo largo del tiempo.
    - Las últimas dos gráficas muestran el espectro de frecuencias de la señal antes y después del filtrado, permitiendo observar cómo el filtro ha afectado las componentes de frecuencia de la señal. Utiliza la función `compute_fft_pos` para obtener el espectro de frecuencias.

### Interfaz de usuario con Streamlit

La interfaz de usuario se crea utilizando Streamlit, una biblioteca que facilita la creación de aplicaciones web.

**Selección de fuente de audio**

Permite elegir entre `Nota sintética`, `Cargar WAV` o `Micrófono en (vivo)`. Donde cada fuente tiene su propios controles que aparecen dinámicamente según la selección.

**Configuracion de la frecuencia de muestreo**

Permite seleccionar la frecuencia de muestreo deseada para el procesamiento de audio, por defecto para microfo es de 48kHz.

**Parámetros de la fuente**

Nota sintética: Permite seleccionar la frecuencia de la nota (f0) y la duración de la nota (dur), el número de armónicos añadidos (harms) y un checkbox para añadir ruido blanco con un SNR especificado usando input de tipo slider.

Cargar WAV: Permite cargar un archivo WAV desde el sistema de archivos local, y un checkbox para añadir ruido blanco con un SNR especificado usando input de tipo slider.

Micrófono en (vivo): Permite capturar audio en tiempo real desde el micrófono, con un checkbox para añadir ruido blanco con un SNR especificado usando input de tipo slider.

**Configuración del filtro**

Permite seleccionar el tipo de filtro (pasa-bajo, pasa-alto, pasa-banda, rechaza-banda) y  la implementación (FIR o IIR con diferentes tipos).

**Frecuencia de corte**

Permite especificar las frecuencias de corte del filtro según el tipo seleccionado y si solo se denomina `cutoff1`entonces se tiene `Paso-Bajo/alto` y si se tienen `cutoff1` y `cutoff2` entonces se tiene `Paso-Banda/Rechaza-Banda`.

**Parámetros de diseño**

Tenemos diferentes parámetros como el nivel de selectividad del filtro el `order` (orden del filtro) que lo que hace es dejar pasar más o menos frecuencias cercanas a la frecuencia de corte. En los IIR se tienen dos variables que se pueden definir.

- `rp`: Rizado en la banda pasante (solo para Chebyshev I).

- `rs`: Rizado en la banda de rechazo (solo para Chebyshev II).

**Parámetros de FIR**

Permite seleccionar el número de taps (coeficientes del filtro) y la ventana a utilizar para el diseño del filtro FIR, la ventana admite diferentes tipos como Hamming, Hann, Blackman y Bartlett.

**Aplicación del filtro**

Aquí se añade ruido a la señal si esta activado, llamando a `apply_filter` para diseñar y aplicar el filtro según los parámetros seleccionados. Luego se grafica la señal original (con ruido si se añadió) y la señal filtrada utilizando `plot_signals_like_example`.

**Microfono en vivo**

Se utiliza un componente que es `st_audiorec` para capturar audio en tiempo real desde el micrófono del usuario. Este componente permite grabar audio directamente desde la interfaz web y luego procesarlo de la misma manera que las otras fuentes de audio. Se lee la señal usando `sf.read` y se convierte en `mono` y se normaliza antes de aplicar el filtro y graficar los resultados.

Luego se añade ruido (opcionalemente) y se aplica el filtro seleccionado a la señal de audio. Finalmente, se grafican las señales originales y filtradas junto con sus espectros de frecuencia para comparar los efectos del filtrado. También, te permite descargar la señal filtrada como un archivo WAV.